In [3]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder

In [5]:
df_train_labels = pd.read_csv('train_labels.csv')
df_train_sequences = pd.read_csv('train_sequences.csv')

In [6]:
df_train_labels.head()

,ID,resname,resid,x_1,y_1,z_1
0,1SCL_A_1,G,1,13.760,-25.974001,0.102
1,1SCL_A_2,G,2,9.310,-29.638000,2.669
2,1SCL_A_3,G,3,5.529,-27.813000,5.878
3,1SCL_A_4,U,4,2.678,-24.900999,9.793
4,1SCL_A_5,G,5,1.827,-20.136000,11.793


In [7]:
df_train_sequences.head()

,target_id,sequence,temporal_cutoff,description,all_sequences
0,1SCL_A,GGGUGCUCAGUACGAGAGGAACCGCACCC,1995-01-26,"THE SARCIN-RICIN LOOP, A MODULAR RNA",>1SCL_1|Chain A|RNA SARCIN-RICIN LOOP|Rattus n...
1,1RNK_A,GGCGCAGUGGGCUAGCGCCACUCAAAAGGCCCAU,1995-02-27,THE STRUCTURE OF AN RNA PSEUDOKNOT THAT CAUSES...,>1RNK_1|Chain A|RNA PSEUDOKNOT|null\nGGCGCAGUG...
2,1RHT_A,GGGACUGACGAUCACGCAGUCUAU,1995-06-03,24-MER RNA HAIRPIN COAT PROTEIN BINDING SITE F...,>1RHT_1|Chain A|RNA (5'-R(P*GP*GP*GP*AP*CP*UP*...
3,1HLX_A,GGGAUAACUUCGGUUGUCCC,1995-09-15,P1 HELIX NUCLEIC ACIDS (DNA/RNA) RIBONUCLEIC ACID,>1HLX_1|Chain A|RNA (5'-R(*GP*GP*GP*AP*UP*AP*A...
4,1HMH_E,GGCGACCCUGAUGAGGCCGAAAGGCCGAAACCGU,1995-12-07,THREE-DIMENSIONAL STRUCTURE OF A HAMMERHEAD RI...,">1HMH_1|Chains A, C, E|HAMMERHEAD RIBOZYME-RNA..."


In [8]:
def get_xyz_dataframe(df):
    # Extract unique ID (first 4 characters of 'ID')
    df["Unique_ID"] = df['ID'].str.extract(r'^([^_]+_[^_]+)', expand=False)

    # Create a dictionary to store results
    data = []

    # Iterate over each unique prefix and extract flattened x, y, z coordinates
    for unique_id in df["Unique_ID"].unique():
        filtered_df = df[df["Unique_ID"] == unique_id]
        xyz_list = filtered_df.iloc[:, 3:6].values.tolist()
        data.append([unique_id, xyz_list])



    # Convert to a DataFrame
    xyz_df = pd.DataFrame(data, columns=["Unique_ID", "Coordinates"])

    return xyz_df

#Assuming 'df' is your original DataFrame
xyz_df = get_xyz_dataframe(df_train_labels)

# Display the resulting DataFrame
print(xyz_df.iloc[0])

Unique_ID                                                 1SCL_A
Coordinates    [[13.760000228881836, -25.974000930786133, 0.1...
Name: 0, dtype: object


In [9]:
pd.set_option("display.max_colwidth", None)

In [10]:
df = df_train_labels.groupby(df_train_labels['ID'].str[:4])
df.head()

,ID,resname,resid,x_1,y_1,z_1,Unique_ID
0,1SCL_A_1,G,1,13.760000,-25.974001,0.102000,1SCL_A
1,1SCL_A_2,G,2,9.310000,-29.638000,2.669000,1SCL_A
2,1SCL_A_3,G,3,5.529000,-27.813000,5.878000,1SCL_A
3,1SCL_A_4,U,4,2.678000,-24.900999,9.793000,1SCL_A
4,1SCL_A_5,G,5,1.827000,-20.136000,11.793000,1SCL_A
...,...,...,...,...,...,...,...
137009,8Z1F_T_1,G,1,103.195999,112.250999,104.455002,8Z1F_T
137010,8Z1F_T_2,G,2,107.467003,108.984001,106.205002,8Z1F_T
137011,8Z1F_T_3,U,3,111.919998,107.942001,109.775002,8Z1F_T
137012,8Z1F_T_4,A,4,114.685997,108.813004,114.404999,8Z1F_T


In [11]:
def one_hot_encode(seq):
    mapping = dict(zip("ACGU", range(4)))
    seq2 = [mapping[i] for i in seq]
    return_seq = np.eye(4, dtype=int)[seq2]
    if len(seq) < 20:
      end = np.zeros((4,20 - len(seq)))
      return_seq = np.append(return_seq,end)
    return return_seq

In [12]:
df_train_labels[(df_train_labels['resname'].str.contains('-')) | (df_train_labels['resname'].str.contains('X'))]

,ID,resname,resid,x_1,y_1,z_1,Unique_ID
87426,6WB1_C_53,-,53,146.863998,162.598007,114.521004,6WB1_C
87427,6WB1_C_54,-,54,148.063995,168.145004,113.511002,6WB1_C
94043,6Y0C_IN1_15,-,15,255.529999,194.612000,232.932999,6Y0C_IN1
94044,6Y0C_IN1_16,-,16,254.729996,200.151001,233.839005,6Y0C_IN1
113186,7SLP_R_1,X,1,NaN,NaN,NaN,7SLP_R
130032,8H6E_4A_1,X,1,NaN,NaN,NaN,8H6E_4A


In [13]:
x = df_train_sequences[df_train_sequences['target_id'].str.contains('7SLP_R')]

In [14]:
x['description'].iloc[0]

'Cryo-EM structure of 7SK core RNP with linear RNA'

In [15]:
x['sequence'].iloc[0]

'XGAUGUGAGGGCGACUUCGGUCCUCCCUCACCGCUCCAUGUGCGAAAUGAGGCGCUGCAUGUGGCAGUCUGCCUUUCUUUU'

# Remove Invalid rows

In [16]:
df_train_labels = df_train_labels[~(df_train_labels['resname'].str.contains('-')) & ~(df_train_labels['resname'].str.contains('X'))]
df_train_labels.head()

,ID,resname,resid,x_1,y_1,z_1,Unique_ID
0,1SCL_A_1,G,1,13.760,-25.974001,0.102,1SCL_A
1,1SCL_A_2,G,2,9.310,-29.638000,2.669,1SCL_A
2,1SCL_A_3,G,3,5.529,-27.813000,5.878,1SCL_A
3,1SCL_A_4,U,4,2.678,-24.900999,9.793,1SCL_A
4,1SCL_A_5,G,5,1.827,-20.136000,11.793,1SCL_A


In [17]:
df_train_labels.to_csv('new_train_labels.csv')

In [18]:
df_train_sequences = df_train_sequences[~(df_train_sequences['sequence'].str.contains('-')) & ~(df_train_sequences['sequence'].str.contains('X'))]
df_train_sequences.head()

,target_id,sequence,temporal_cutoff,description,all_sequences
0,1SCL_A,GGGUGCUCAGUACGAGAGGAACCGCACCC,1995-01-26,"THE SARCIN-RICIN LOOP, A MODULAR RNA",>1SCL_1|Chain A|RNA SARCIN-RICIN LOOP|Rattus norvegicus (10116)\nGGGUGCUCAGUACGAGAGGAACCGCACCC\n
1,1RNK_A,GGCGCAGUGGGCUAGCGCCACUCAAAAGGCCCAU,1995-02-27,THE STRUCTURE OF AN RNA PSEUDOKNOT THAT CAUSES EFFICIENT FRAMESHIFTING IN MOUSE MAMMARY TUMOR VIRUS,>1RNK_1|Chain A|RNA PSEUDOKNOT|null\nGGCGCAGUGGGCUAGCGCCACUCAAAAGGCCCAU\n
2,1RHT_A,GGGACUGACGAUCACGCAGUCUAU,1995-06-03,"24-MER RNA HAIRPIN COAT PROTEIN BINDING SITE FOR BACTERIOPHAGE R17 (NMR, MINIMIZED AVERAGE STRUCTURE)",>1RHT_1|Chain A|RNA (5'-R(P*GP*GP*GP*AP*CP*UP*GP*AP*CP*GP*AP*UP*CP*AP*CP*GP*CP*AP*GP*UP*CP*UP*AP*U)-3')|null\nGGGACUGACGAUCACGCAGUCUAU\n
3,1HLX_A,GGGAUAACUUCGGUUGUCCC,1995-09-15,P1 HELIX NUCLEIC ACIDS (DNA/RNA) RIBONUCLEIC ACID,>1HLX_1|Chain A|RNA (5'-R(*GP*GP*GP*AP*UP*AP*AP*CP*UP*UP*CP*GP*GP*UP*UP*GP*UP*CP*CP*C)-3')|null\nGGGAUAACUUCGGUUGUCCC\n
4,1HMH_E,GGCGACCCUGAUGAGGCCGAAAGGCCGAAACCGU,1995-12-07,THREE-DIMENSIONAL STRUCTURE OF A HAMMERHEAD RIBOZYME,">1HMH_1|Chains A, C, E|HAMMERHEAD RIBOZYME-RNA STRAND|\nGGCGACCCUGAUGAGGCCGAAAGGCCGAAACCGU\n>1HMH_2|Chains B, D, F|HAMMERHEAD RIBOZYME-DNA STRAND|\nACGGTCGGTCGCC\n"


In [19]:
df_train_sequences.to_csv('new_train_sequences.csv')

# Try OHE with clean data

In [20]:
df_train_labels['ohe'] = df_train_labels['resname'].apply(one_hot_encode)
df_train_labels.head()

/tmp/ipykernel_4165/84251357.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train_labels['ohe'] = df_train_labels['resname'].apply(one_hot_encode)


,ID,resname,resid,x_1,y_1,z_1,Unique_ID,ohe
0,1SCL_A_1,G,1,13.760,-25.974001,0.102,1SCL_A,"[0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"
1,1SCL_A_2,G,2,9.310,-29.638000,2.669,1SCL_A,"[0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"
2,1SCL_A_3,G,3,5.529,-27.813000,5.878,1SCL_A,"[0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"
3,1SCL_A_4,U,4,2.678,-24.900999,9.793,1SCL_A,"[0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"
4,1SCL_A_5,G,5,1.827,-20.136000,11.793,1SCL_A,"[0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"


In [21]:
df_train_sequences['ohe'] = df_train_sequences['sequence'].apply(one_hot_encode)
df_train_sequences.head()

,target_id,sequence,temporal_cutoff,description,all_sequences,ohe
0,1SCL_A,GGGUGCUCAGUACGAGAGGAACCGCACCC,1995-01-26,"THE SARCIN-RICIN LOOP, A MODULAR RNA",>1SCL_1|Chain A|RNA SARCIN-RICIN LOOP|Rattus norvegicus (10116)\nGGGUGCUCAGUACGAGAGGAACCGCACCC\n,"[[0, 0, 1, 0], [0, 0, 1, 0], [0, 0, 1, 0], [0, 0, 0, 1], [0, 0, 1, 0], [0, 1, 0, 0], [0, 0, 0, 1], [0, 1, 0, 0], [1, 0, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1], [1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [1, 0, 0, 0], [0, 0, 1, 0], [1, 0, 0, 0], [0, 0, 1, 0], [0, 0, 1, 0], [1, 0, 0, 0], [1, 0, 0, 0], [0, 1, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 1, 0, 0], [1, 0, 0, 0], [0, 1, 0, 0], [0, 1, 0, 0], [0, 1, 0, 0]]"
1,1RNK_A,GGCGCAGUGGGCUAGCGCCACUCAAAAGGCCCAU,1995-02-27,THE STRUCTURE OF AN RNA PSEUDOKNOT THAT CAUSES EFFICIENT FRAMESHIFTING IN MOUSE MAMMARY TUMOR VIRUS,>1RNK_1|Chain A|RNA PSEUDOKNOT|null\nGGCGCAGUGGGCUAGCGCCACUCAAAAGGCCCAU\n,"[[0, 0, 1, 0], [0, 0, 1, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 1, 0, 0], [1, 0, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1], [0, 0, 1, 0], [0, 0, 1, 0], [0, 0, 1, 0], [0, 1, 0, 0], [0, 0, 0, 1], [1, 0, 0, 0], [0, 0, 1, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 1, 0, 0], [0, 1, 0, 0], [1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 0, 1], [0, 1, 0, 0], [1, 0, 0, 0], [1, 0, 0, 0], [1, 0, 0, 0], [1, 0, 0, 0], [0, 0, 1, 0], [0, 0, 1, 0], [0, 1, 0, 0], [0, 1, 0, 0], [0, 1, 0, 0], [1, 0, 0, 0], [0, 0, 0, 1]]"
2,1RHT_A,GGGACUGACGAUCACGCAGUCUAU,1995-06-03,"24-MER RNA HAIRPIN COAT PROTEIN BINDING SITE FOR BACTERIOPHAGE R17 (NMR, MINIMIZED AVERAGE STRUCTURE)",>1RHT_1|Chain A|RNA (5'-R(P*GP*GP*GP*AP*CP*UP*GP*AP*CP*GP*AP*UP*CP*AP*CP*GP*CP*AP*GP*UP*CP*UP*AP*U)-3')|null\nGGGACUGACGAUCACGCAGUCUAU\n,"[[0, 0, 1, 0], [0, 0, 1, 0], [0, 0, 1, 0], [1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0], [1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [1, 0, 0, 0], [0, 0, 0, 1], [0, 1, 0, 0], [1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 1, 0, 0], [1, 0, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1], [0, 1, 0, 0], [0, 0, 0, 1], [1, 0, 0, 0], [0, 0, 0, 1]]"
3,1HLX_A,GGGAUAACUUCGGUUGUCCC,1995-09-15,P1 HELIX NUCLEIC ACIDS (DNA/RNA) RIBONUCLEIC ACID,>1HLX_1|Chain A|RNA (5'-R(*GP*GP*GP*AP*UP*AP*AP*CP*UP*UP*CP*GP*GP*UP*UP*GP*UP*CP*CP*C)-3')|null\nGGGAUAACUUCGGUUGUCCC\n,"[[0, 0, 1, 0], [0, 0, 1, 0], [0, 0, 1, 0], [1, 0, 0, 0], [0, 0, 0, 1], [1, 0, 0, 0], [1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 0, 1], [0, 0, 0, 1], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 1, 0], [0, 0, 0, 1], [0, 0, 0, 1], [0, 0, 1, 0], [0, 0, 0, 1], [0, 1, 0, 0], [0, 1, 0, 0], [0, 1, 0, 0]]"
4,1HMH_E,GGCGACCCUGAUGAGGCCGAAAGGCCGAAACCGU,1995-12-07,THREE-DIMENSIONAL STRUCTURE OF A HAMMERHEAD RIBOZYME,">1HMH_1|Chains A, C, E|HAMMERHEAD RIBOZYME-RNA STRAND|\nGGCGACCCUGAUGAGGCCGAAAGGCCGAAACCGU\n>1HMH_2|Chains B, D, F|HAMMERHEAD RIBOZYME-DNA STRAND|\nACGGTCGGTCGCC\n","[[0, 0, 1, 0], [0, 0, 1, 0], [0, 1, 0, 0], [0, 0, 1, 0], [1, 0, 0, 0], [0, 1, 0, 0], [0, 1, 0, 0], [0, 1, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0], [1, 0, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0], [1, 0, 0, 0], [0, 0, 1, 0], [0, 0, 1, 0], [0, 1, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [1, 0, 0, 0], [1, 0, 0, 0], [1, 0, 0, 0], [0, 0, 1, 0], [0, 0, 1, 0], [0, 1, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [1, 0, 0, 0], [1, 0, 0, 0], [1, 0, 0, 0], [0, 1, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]]"


In [22]:
df_train_sequences['length'] = df_train_sequences['sequence'].str.len()
df_train_less_than_20 = df_train_sequences[df_train_sequences['length'] >= 20]
df_train_less_than_20.shape

(683, 7)

In [23]:
df_train_less_than_20.head(1)

,target_id,sequence,temporal_cutoff,description,all_sequences,ohe,length
0,1SCL_A,GGGUGCUCAGUACGAGAGGAACCGCACCC,1995-01-26,"THE SARCIN-RICIN LOOP, A MODULAR RNA",>1SCL_1|Chain A|RNA SARCIN-RICIN LOOP|Rattus norvegicus (10116)\nGGGUGCUCAGUACGAGAGGAACCGCACCC\n,"[[0, 0, 1, 0], [0, 0, 1, 0], [0, 0, 1, 0], [0, 0, 0, 1], [0, 0, 1, 0], [0, 1, 0, 0], [0, 0, 0, 1], [0, 1, 0, 0], [1, 0, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1], [1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [1, 0, 0, 0], [0, 0, 1, 0], [1, 0, 0, 0], [0, 0, 1, 0], [0, 0, 1, 0], [1, 0, 0, 0], [1, 0, 0, 0], [0, 1, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 1, 0, 0], [1, 0, 0, 0], [0, 1, 0, 0], [0, 1, 0, 0], [0, 1, 0, 0]]",29


In [24]:
df_train_less_than_20['ohe'].head()

0                                                                          [[0, 0, 1, 0], [0, 0, 1, 0], [0, 0, 1, 0], [0, 0, 0, 1], [0, 0, 1, 0], [0, 1, 0, 0], [0, 0, 0, 1], [0, 1, 0, 0], [1, 0, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1], [1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [1, 0, 0, 0], [0, 0, 1, 0], [1, 0, 0, 0], [0, 0, 1, 0], [0, 0, 1, 0], [1, 0, 0, 0], [1, 0, 0, 0], [0, 1, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 1, 0, 0], [1, 0, 0, 0], [0, 1, 0, 0], [0, 1, 0, 0], [0, 1, 0, 0]]
1    [[0, 0, 1, 0], [0, 0, 1, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 1, 0, 0], [1, 0, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1], [0, 0, 1, 0], [0, 0, 1, 0], [0, 0, 1, 0], [0, 1, 0, 0], [0, 0, 0, 1], [1, 0, 0, 0], [0, 0, 1, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 1, 0, 0], [0, 1, 0, 0], [1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 0, 1], [0, 1, 0, 0], [1, 0, 0, 0], [1, 0, 0, 0], [1, 0, 0, 0], [1, 0, 0, 0], [0, 0, 1, 0], [0, 0, 1, 0], [0, 1, 0, 0], [0, 1, 0, 0], [0, 1, 0, 0], [1, 0, 0, 0], [0, 0, 0, 1]]
2                                   

In [25]:
from sklearn.model_selection import train_test_split

In [43]:
x = df_train_labels[df_train_labels['x_1'].isnull()]
x.shape

(6143, 8)

In [44]:
x.shape[0] / df_train_labels.shape[0]

0.044810305713806356